<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/10_image_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 画像検索 / CLIP + pytidb (auto-embedding 版)

このノートブックは **pytidb シリーズの第 10 回** です。マルチモーダルの埋め込みモデルである、OpenAI の CLIP (ViT-B/32) を pytidb の **auto-embedding API** に接続し、TiDB で画像検索を行います。

この演習は [pytidb `examples/image_search/app.py`](https://github.com/pingcap/pytidb/blob/main/examples/image_search/app.py) の構成を、Google Colab で動くように CLIP 置き換えで再構築したものです。

## 学習目標
- `BaseEmbeddingFunction` を継承し、`source_type="text"` / `"image"` の両方に対応したマルチモーダル埋め込みクラスを書く
- `VectorField(source_field="image_uri", source_type="image")` で **画像の auto-embedding 列**を宣言する
- 画像は **`file://` URI** として TiDB に保存し、pytidb に embed を任せる (`table.bulk_insert` のみで OK)
- 同じ VectorField に対して **テキスト→画像 検索**と **画像→画像 検索** を両方実行する
  - pytidb は `source_type="image"` を常に `get_query_embedding` に渡すが、我々の wrapper 側で入力型から auto-dispatch する

## 注意
- CLIP ViT-B/32 は **英語**のテキストに最適化されたモデルです。クエリ文は英語で書いてください
- 初回のみ CLIPモデルのダウンロード が走ります (約 150 MB)、Colab の無料枠 (CPU ランタイム) で数十秒〜1 分程度
- API キーは不要 (CLIP はローカル推論)

前提: `09_custom_embedding.ipynb` を実行済みだとカスタムモデルの利用方法が分かり、理解がスムーズです。


## 1. 依存パッケージのインストール

追加で `transformers`・`datasets`・`pillow` を入れます (`torch` は Colab に同梱)。


In [ ]:
!pip install -q pytidb transformers datasets pillow


## 2. TiDB Cloud Zero の払い出し


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-image-search")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 3. URI / PIL 両対応のマルチモーダル CLIP クラス

`BaseEmbeddingFunction` を継承し、**auto-embedding のインターフェース**に合わせます:

- `get_source_embedding(s)`: insert 時に `image_uri` 列の文字列が渡ってくるので `source_type="image"` で画像として読み込む
- `get_query_embedding`: 検索時、`source_type` は VectorField の宣言値 (`"image"`) が固定で渡るので、**実入力の型から auto-dispatch** (PIL.Image / URI 文字列 → 画像エンコーダ、普通の文字列 → テキストエンコーダ)

`_load_image()` は `file://` / `http(s)://` / ローカルパス / PIL.Image のどれでも受けて PIL に正規化します。


In [ ]:
import io
import urllib.request
from pathlib import Path
from typing import Any, List, Optional

import requests
import torch
from PIL import Image
from PIL.Image import Image as PILImage
from transformers import CLIPModel, CLIPProcessor
from pytidb.embeddings.base import BaseEmbeddingFunction

CLIP_MODEL = "openai/clip-vit-base-patch32"
CLIP_DIM = 512
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif")


def _load_image(src: Any) -> PILImage:
    """PIL.Image / Path / str (file:// / http(s):// / plain path) を RGB PIL に正規化"""
    if isinstance(src, PILImage):
        return src.convert("RGB")
    if isinstance(src, Path):
        return Image.open(src).convert("RGB")
    if isinstance(src, str):
        if src.startswith("file://"):
            path = urllib.request.url2pathname(src[len("file://"):])
            return Image.open(path).convert("RGB")
        if src.startswith(("http://", "https://")):
            data = requests.get(src, timeout=10).content
            return Image.open(io.BytesIO(data)).convert("RGB")
        return Image.open(src).convert("RGB")
    raise TypeError(f"unsupported image source: {type(src).__name__}")


def _is_image_like(x: Any) -> bool:
    """入力が画像と判断できるかを判定する。質問文は文字のケースと画像のケースがあり、どちらも同様に扱うため"""
    if isinstance(x, (PILImage, Path)):
        return True
    if isinstance(x, str):
        if x.startswith(("file://", "http://", "https://")):
            return True
        if any(x.lower().endswith(ext) for ext in IMAGE_EXTS):
            return True
    return False


class CLIPEmbedding(BaseEmbeddingFunction):
    model_config = {"arbitrary_types_allowed": True}

    def __init__(self, model_name: str = CLIP_MODEL, **kwargs):
        super().__init__(
            provider="clip",
            model_name=model_name,
            dimensions=CLIP_DIM,
            use_server=False,
            **kwargs,
        )
        self.__dict__["_model"] = CLIPModel.from_pretrained(model_name)
        self.__dict__["_processor"] = CLIPProcessor.from_pretrained(model_name)

    @staticmethod
    def _to_tensor(out):
        # transformers のバージョン差を吸収 (tensor / BaseModelOutputWithPooling 両対応)
        if hasattr(out, "cpu"):
            return out
        for attr in ("text_embeds", "image_embeds", "pooler_output", "last_hidden_state"):
            v = getattr(out, attr, None)
            if v is not None and hasattr(v, "cpu"):
                return v
        raise TypeError(f"Cannot extract tensor from CLIP output: {type(out).__name__}")

    def _encode_text(self, texts: List[str]) -> List[List[float]]:
        proc = self.__dict__["_processor"]
        model = self.__dict__["_model"]
        with torch.no_grad():
            inputs = proc(text=texts, return_tensors="pt", padding=True, truncation=True)
            features = self._to_tensor(model.get_text_features(**inputs))
        return [row.tolist() for row in features.cpu().numpy()]

    def _encode_images(self, images) -> List[List[float]]:
        proc = self.__dict__["_processor"]
        model = self.__dict__["_model"]
        pil_images = [_load_image(im) for im in images]
        with torch.no_grad():
            inputs = proc(images=pil_images, return_tensors="pt")
            features = self._to_tensor(model.get_image_features(**inputs))
        return [row.tolist() for row in features.cpu().numpy()]

    # pytidb が呼ぶ 3 つのエントリポイント

    def get_source_embedding(self, source: Any, source_type: Optional[str] = "text", **kwargs) -> List[float]:
        # insert 時: source は image_uri 列の文字列 (VectorField.source_type="image" で固定)
        if source_type == "image":
            return self._encode_images([source])[0]
        return self._encode_text([str(source)])[0]

    def get_source_embeddings(self, sources: List[Any], source_type: Optional[str] = "text", **kwargs) -> List[List[float]]:
        if source_type == "image":
            return self._encode_images(sources)
        return self._encode_text([str(s) for s in sources])

    def get_query_embedding(self, query: Any, source_type: Optional[str] = "text", **kwargs) -> List[float]:
        # 検索時: source_type は "image" で常に固定なので実入力から auto-dispatch する
        if _is_image_like(query):
            return self._encode_images([query])[0]
        return self._encode_text([str(query)])[0]


embed_fn = CLIPEmbedding()
print("CLIP ready  dim =", embed_fn.dimensions)


## 4. テーブル定義 (auto-embedding 列)

画像検索のためのテーブルを作ります。`image_uri` 列に `file://` URI 文字列を入れ、`image_vec` は **`source_field="image_uri"`** + **`source_type="image"`** を指定したベクトル型とします。これで、insert 時に `image_uri` 列の値が `source_type="image"` として `embed_fn.get_source_embedding` に渡るようになります。

In [ ]:
from pytidb import TiDBClient
from pytidb.datatype import TEXT
from pytidb.schema import Field, TableModel

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database="image_search_demo",
    ensure_db=True,
)


class ImageRecord(TableModel):
    __tablename__ = "image_records"
    __table_args__ = {"extend_existing": True}

    id: int = Field(primary_key=True)
    label: str = Field()
    image_uri: str = Field(sa_type=TEXT)                    # file:// URI を格納
    image_vec: list[float] = embed_fn.VectorField(
        source_field="image_uri",
        source_type="image",
    )


table = db.create_table(schema=ImageRecord, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


## 5. サンプル画像をロード + 画像をローカル保存

`datasets` から ImageNet tiny を 20 枚取得し、Colab のローカルに `.jpg` として保存して `file://` URI を作ります。


In [ ]:
import os
import datasets

IMG_DIR = "/content/img_pool"
os.makedirs(IMG_DIR, exist_ok=True)

ds = datasets.load_dataset("theodor1289/imagenet-1k_tiny", split="train")

records = []     # [(id, label, image_uri, PIL.Image)] for preview/reference
for i, row in enumerate(ds):
    if i >= 20:
        break
    img = row["image"].convert("RGB")
    label = str(row.get("label", row.get("fine_label", "unknown")))
    path = os.path.abspath(os.path.join(IMG_DIR, f"{i:03d}.jpg"))
    img.save(path, "JPEG")
    uri = f"file://{path}"
    records.append((i + 1, label, uri, img))

print(f"saved {len(records)} images to {IMG_DIR}")
print("example URI:", records[0][2])


### 画像をインラインで確認する


In [ ]:
import matplotlib.pyplot as plt

def show_grid(images, titles=None, cols=5, width=2):
    n = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * width, rows * width))
    axes = axes.flatten() if n > 1 else [axes]
    for ax in axes:
        ax.axis("off")
    for i, img in enumerate(images):
        axes[i].imshow(img)
        if titles:
            axes[i].set_title(titles[i], fontsize=9)
    plt.tight_layout()
    plt.show()


show_grid(
    [r[3] for r in records],
    titles=[f"#{r[0]} {r[1]}" for r in records],
)


## 6. bulk_insert で自動埋め込み

`ImageRecord(id, label, image_uri)` を 20 件まとめて `table.bulk_insert(...)` に渡すと、pytidb が内部で
`embed_fn.get_source_embeddings([uri, uri, ...], source_type="image")` を呼び、画像を CLIP で 512 次元にしてから `image_vec` 列に書き込みます。
**手動でベクトル化する必要はありません。**


In [ ]:
rows = [
    ImageRecord(id=rid, label=label, image_uri=uri)
    for (rid, label, uri, _) in records
]
table.bulk_insert(rows)
print(f"投入完了: {table.rows()} 件")


## 7. テキストから画像を検索する

`table.search(QUERY)` を呼ぶだけです。pytidb は自動で `get_query_embedding` を呼び、`CLIPEmbedding` のヘルパー関数が文字列と判定し、テキストエンコーダに回してくれます。CLIP のテキストエンコーダは、画像と同じベクトル空間にテキストをマッピングするように訓練されているため、**テキストクエリで画像を検索できます**。


In [ ]:
QUERIES = ["a photo of a dog", "sushi on a plate", "a classic sports car"]

for q in QUERIES:
    hits = table.search(q).limit(5).to_pydantic()
    print(f"\n=== query: {q!r} ===")
    imgs = [_load_image(h.hit.image_uri) for h in hits]
    titles = [f"sim={h.similarity_score:.3f}\n[{h.hit.label}]" for h in hits]
    show_grid(imgs, titles=titles, cols=5)


## 8. 画像から画像を検索する

検索対象にも `table.search(pil_image)` と PIL Image を直接渡せます。`CLIPEmbedding` のヘルパー関数が `PIL.Image` を検出して画像エンコーダを使います。

検索画像は20個の画像から選んでいるため、結果の中に同じ画像を含みます。そのため、top-1 は必ずクエリ自身 (sim≈1.0) になります。


In [ ]:
# データセット先頭の 1 枚をクエリにする
query_image = records[0][3]
print("Query image:")
show_grid([query_image], titles=["query"], cols=1)

hits = table.search(query_image).limit(5).to_pydantic()

print("\n=== top-5 similar images ===")
imgs = [_load_image(h.hit.image_uri) for h in hits]
titles = [f"sim={h.similarity_score:.3f}\n[{h.hit.label}]" for h in hits]
show_grid(imgs, titles=titles, cols=5)


## チャレンジ

- `table.search("...")` / `table.search(pil_image)` のどちらでも **同じ image_vec 列**が使われていることを確認
- `QUERIES` に英語フレーズを追加 / 外部画像 URL (`https://...`) を直接 `table.search(url)` に渡す (wrapper 側で自動ロード)
- `table.search(...).distance_threshold(0.5).limit(10)` で閾値による足切り
- CLIP を **`openai/clip-vit-large-patch14`** (約 900 MB、768 次元) に差し替え → `CLIP_DIM = 768` に注意
